# OmicSage — Phase 1: QC Pipeline

This notebook runs the full QC module on all benchmark datasets and generates HTML reports.

**Pipeline steps covered:**
1. Data ingestion (`ingest.py`) — load raw counts from any format
2. Quality control (`qc.py`) — filter low-quality cells, detect doublets
3. Report generation (`qc_report.py`) — HTML report per dataset

**Datasets:**
- GSE194122 CITE-seq BMMC (`.h5ad`)
- GSE166635 HCC scRNA-seq (10x MTX)

---

## 0. Setup

In [ ]:
import os
import sys

# Make sure we are running from the OmicSage root
# If this cell errors, run: os.chdir('/home/<you>/OmicSage')
assert os.path.exists('pipeline/modules/qc/qc.py'), (
    "Run this notebook from the OmicSage root directory. "
    "Current dir: " + os.getcwd()
)

# Suppress scanpy/anndata verbosity
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import scanpy as sc
sc.settings.verbosity = 1

print('✓ Setup complete')
print(f'  Working directory : {os.getcwd()}')
print(f'  Python            : {sys.version.split()[0]}')
print(f'  Scanpy            : {sc.__version__}')

In [ ]:
from pipeline.modules.qc.ingest import load_dataset
from pipeline.modules.qc.qc import run_qc

# Output directory for reports
os.makedirs('reports', exist_ok=True)

print('✓ Imports OK')

---
## 1. GSE194122 — CITE-seq BMMC (`.h5ad`)

NeurIPS 2021 multimodal benchmark dataset.  
90,261 cells × 14,087 features (RNA + ADT).  
Ground truth: `obs['GEX_pct_counts_mt']`, `obs['cell_type']`

### 1.1 Load data

In [ ]:
CITE_PATH = 'data/benchmark/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad'

adata_cite = load_dataset(CITE_PATH, sample_name='GSE194122_BMMC_CITE')

In [ ]:
# Quick sanity check
print(f'Cells    : {adata_cite.n_obs:,}')
print(f'Features : {adata_cite.n_vars:,}')
print(f'Layers   : {list(adata_cite.layers.keys())}')
print(f'X dtype  : {adata_cite.X.dtype}')
print()
print('obs columns (first 10):')
print(list(adata_cite.obs.columns[:10]))
print()
print('Ground truth MT% available:', 'GEX_pct_counts_mt' in adata_cite.obs.columns)

### 1.2 Run QC

In [ ]:
adata_cite_filtered, metrics_cite = run_qc(
    adata_cite,
    min_genes=200,
    max_genes=6000,
    max_mt_pct=20.0,
    remove_doublets=True,
    generate_report=True,
    report_path='reports/qc_GSE194122_cite.html',
    sample_name='GSE194122_BMMC_CITE',
)

### 1.3 QC Summary

In [ ]:
m = metrics_cite
pass_rate = 100 * m['n_cells_output'] / m['n_cells_input']

print('─' * 45)
print(f"  {'Cells input':<25} {m['n_cells_input']:>10,}")
print(f"  {'Cells kept':<25} {m['n_cells_output']:>10,}  ({pass_rate:.1f}%)")
print(f"  {'Cells removed':<25} {m['n_cells_removed']:>10,}")
print('─' * 45)
print(f"  {'Removed (low genes)':<25} {m['n_removed_low_genes']:>10,}")
print(f"  {'Removed (high genes)':<25} {m['n_removed_high_genes']:>10,}")
print(f"  {'Removed (high MT%)':<25} {m['n_removed_high_mt']:>10,}")
print(f"  {'Removed (doublets)':<25} {m['n_removed_doublets']:>10,}")
print('─' * 45)
print(f"  {'Median genes/cell':<25} {m['median_genes_per_cell']:>10.0f}")
print(f"  {'Median UMI/cell':<25} {m['median_umi_per_cell']:>10.0f}")
print(f"  {'Median MT%':<25} {m['median_mt_pct']:>10.2f}%")
print(f"  {'MT genes detected':<25} {m['n_mt_genes']:>10,}")
print('─' * 45)

### 1.4 Validate MT% against ground truth

In [ ]:
import numpy as np
import scipy.sparse as sp

# Subset to GEX features only for fair MT% comparison
if 'feature_types' in adata_cite.var.columns:
    rna_mask = adata_cite.var['feature_types'].astype(str).isin(['Gene Expression', 'GEX'])
    adata_rna = adata_cite[:, rna_mask].copy()
else:
    adata_rna = adata_cite.copy()

# Remove zero-count cells
cell_totals = np.asarray(adata_rna.X.sum(axis=1)).flatten()
adata_rna = adata_rna[cell_totals > 0].copy()

adata_rna_f, _ = run_qc(
    adata_rna,
    min_genes=1, max_genes=999_999, max_mt_pct=100.0,
    remove_doublets=False, generate_report=False,
)

our_mt = adata_rna_f.obs['pct_counts_mt'].values
gt_mt  = adata_rna_f.obs['GEX_pct_counts_mt'].values
valid  = np.isfinite(our_mt) & np.isfinite(gt_mt)
corr   = float(np.corrcoef(our_mt[valid], gt_mt[valid])[0, 1])
mae    = float(np.mean(np.abs(our_mt[valid] - gt_mt[valid])))

status = '✓ PASS' if corr > 0.99 else '✗ FAIL'
print(f'MT% correlation vs ground truth : {corr:.4f}  {status} (threshold > 0.99)')
print(f'MT% mean absolute error         : {mae:.4f}%  (threshold < 0.5%)')

---
## 2. GSE166635 — HCC scRNA-seq (10x MTX)

Wang et al. 2025 hepatocellular carcinoma dataset.  
Primary benchmark for Phase 1 milestone.

### 2.1 Load data

In [ ]:
HCC_PATH = 'data/test/GSE166635/HCC1'

adata_hcc = load_dataset(HCC_PATH, sample_name='GSE166635_HCC1')

In [ ]:
print(f'Cells    : {adata_hcc.n_obs:,}')
print(f'Genes    : {adata_hcc.n_vars:,}')
print(f'X dtype  : {adata_hcc.X.dtype}')

### 2.2 Run QC

In [ ]:
adata_hcc_filtered, metrics_hcc = run_qc(
    adata_hcc,
    min_genes=200,
    max_genes=6000,
    max_mt_pct=20.0,
    remove_doublets=True,
    generate_report=True,
    report_path='reports/qc_GSE166635_HCC1.html',
    sample_name='GSE166635_HCC1',
)

### 2.3 QC Summary

In [ ]:
m = metrics_hcc
pass_rate = 100 * m['n_cells_output'] / m['n_cells_input']

print('─' * 45)
print(f"  {'Cells input':<25} {m['n_cells_input']:>10,}")
print(f"  {'Cells kept':<25} {m['n_cells_output']:>10,}  ({pass_rate:.1f}%)")
print(f"  {'Cells removed':<25} {m['n_cells_removed']:>10,}")
print('─' * 45)
print(f"  {'Removed (low genes)':<25} {m['n_removed_low_genes']:>10,}")
print(f"  {'Removed (high genes)':<25} {m['n_removed_high_genes']:>10,}")
print(f"  {'Removed (high MT%)':<25} {m['n_removed_high_mt']:>10,}")
print(f"  {'Removed (doublets)':<25} {m['n_removed_doublets']:>10,}")
print('─' * 45)
print(f"  {'Median genes/cell':<25} {m['median_genes_per_cell']:>10.0f}")
print(f"  {'Median UMI/cell':<25} {m['median_umi_per_cell']:>10.0f}")
print(f"  {'Median MT%':<25} {m['median_mt_pct']:>10.2f}%")
print(f"  {'MT genes detected':<25} {m['n_mt_genes']:>10,}")
print('─' * 45)

---
## 3. Reports

Both HTML reports have been written to `reports/`.  
Open them in your browser to review.

In [ ]:
from pathlib import Path

reports = list(Path('reports').glob('qc_*.html'))
print(f'Reports generated: {len(reports)}')
for r in sorted(reports):
    size_kb = r.stat().st_size / 1024
    print(f'  {r}  ({size_kb:.0f} KB)')

---
## 4. Save filtered AnnData objects

Save the QC-filtered objects for use in the next step (normalization).

In [ ]:
os.makedirs('data/processed', exist_ok=True)

adata_cite_filtered.write_h5ad('data/processed/GSE194122_cite_qc.h5ad')
print('✓ Saved: data/processed/GSE194122_cite_qc.h5ad')

adata_hcc_filtered.write_h5ad('data/processed/GSE166635_HCC1_qc.h5ad')
print('✓ Saved: data/processed/GSE166635_HCC1_qc.h5ad')

---
*OmicSage · Phase 1 · QC Module · pipeline/modules/qc/qc.py*